# Kyphosis Prediction Model Training

This notebook trains a Decision Tree Classifier for predicting spinal kyphosis based on patient data.

## Dataset Description
- **Kyphosis**: Target variable ('present' or 'absent')
- **Age**: Patient age in months
- **Number**: Number of vertebrae involved in the operation
- **Start**: Topmost vertebra level operated on

## 1. Import Required Libraries

In [ ]:
#!/usr/bin/env python3
"""
Train a Decision Tree Classifier for Kyphosis prediction.

This notebook loads the kyphosis dataset, trains a model, evaluates it,
and saves the trained model for use by the web application.
"""

import pandas as pd
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
def load_and_explore_data():
    """Load the kyphosis dataset and explore its structure."""
    # Load the dataset
    data_path = os.path.join('..', 'Dataset', 'kyphosis.csv')
    df = pd.read_csv(data_path)
    
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    return df

# Load the data
df = load_and_explore_data()
df.head()

In [ ]:
# Basic dataset information
print("Dataset Info:")
print(df.info())
print("\nDataset Description:")
print(df.describe())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Target variable distribution
print("Target Distribution:")
target_dist = df['Kyphosis'].value_counts()
print(target_dist)
print(f"\nPercentages:")
print((target_dist / len(df) * 100).round(2))

# Plot target distribution
plt.figure(figsize=(8, 6))
df['Kyphosis'].value_counts().plot(kind='bar', color=['lightcoral', 'lightblue'])
plt.title('Distribution of Kyphosis Cases')
plt.xlabel('Kyphosis Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 3. Data Visualization and Analysis

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Age distribution by Kyphosis status
axes[0, 0].hist(df[df['Kyphosis'] == 'present']['Age'], alpha=0.7, label='Present', bins=20, color='lightcoral')
axes[0, 0].hist(df[df['Kyphosis'] == 'absent']['Age'], alpha=0.7, label='Absent', bins=20, color='lightblue')
axes[0, 0].set_title('Age Distribution by Kyphosis Status')
axes[0, 0].set_xlabel('Age (months)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Number of vertebrae distribution
axes[0, 1].hist(df[df['Kyphosis'] == 'present']['Number'], alpha=0.7, label='Present', bins=10, color='lightcoral')
axes[0, 1].hist(df[df['Kyphosis'] == 'absent']['Number'], alpha=0.7, label='Absent', bins=10, color='lightblue')
axes[0, 1].set_title('Number of Vertebrae Distribution')
axes[0, 1].set_xlabel('Number of Vertebrae')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Start vertebra distribution
axes[1, 0].hist(df[df['Kyphosis'] == 'present']['Start'], alpha=0.7, label='Present', bins=15, color='lightcoral')
axes[1, 0].hist(df[df['Kyphosis'] == 'absent']['Start'], alpha=0.7, label='Absent', bins=15, color='lightblue')
axes[1, 0].set_title('Start Vertebra Distribution')
axes[1, 0].set_xlabel('Start Vertebra Level')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# Correlation heatmap
# Create numeric version for correlation
df_numeric = df.copy()
df_numeric['Kyphosis_numeric'] = df['Kyphosis'].map({'present': 1, 'absent': 0})
correlation_matrix = df_numeric[['Age', 'Number', 'Start', 'Kyphosis_numeric']].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

## 4. Data Preparation

In [ ]:
def prepare_data(df):
    """Prepare features and target variables."""
    # Prepare features and target
    X = df[['Age', 'Number', 'Start']]
    y = df['Kyphosis'].map({'present': 1, 'absent': 0})
    
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"\nTarget mapping:")
    print(f"'present' -> 1 (Kyphosis is present)")
    print(f"'absent' -> 0 (Kyphosis is absent)")
    
    # Show class distribution after mapping
    print(f"\nClass distribution after mapping:")
    print(y.value_counts().sort_index())
    
    return X, y

X, y = prepare_data(df)
X.head()

## 5. Model Training and Evaluation

In [ ]:
def train_and_evaluate_model(X, y):
    """Train a Decision Tree classifier and evaluate its performance."""
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set size: {X_train.shape[0]}")
    print(f"Test set size: {X_test.shape[0]}")
    print(f"Training set class distribution:")
    print(y_train.value_counts().sort_index())
    
    # Train the model
    model = DecisionTreeClassifier(
        random_state=42,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced'  # This helps with imbalanced classes
    )
    
    print(f"\nTraining the Decision Tree model...")
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n" + "="*50)
    print(f"MODEL PERFORMANCE METRICS")
    print(f"="*50)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision (weighted): {precision:.4f}")
    print(f"Recall (weighted): {recall:.4f}")
    print(f"F1-Score (weighted): {f1:.4f}")
    
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Absent (0)', 'Present (1)']))
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    # Feature importance
    feature_names = ['Age', 'Number', 'Start']
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\nFeature Importance:")
    print(feature_importance)
    
    return model, X_test, y_test, y_pred, y_pred_proba, feature_importance

model, X_test, y_test, y_pred, y_pred_proba, feature_importance = train_and_evaluate_model(X, y)

## 6. Model Visualization

In [ ]:
# Plot confusion matrix and feature importance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Absent (0)', 'Present (1)'],
            yticklabels=['Absent (0)', 'Present (1)'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Feature Importance
feature_importance.plot(kind='bar', x='feature', y='importance', ax=axes[1], 
                       color='skyblue', legend=False)
axes[1].set_title('Feature Importance')
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Importance')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Model Testing with Sample Predictions

In [ ]:
def test_model_predictions(model, test_cases=None):
    """Test the model with specific cases to ensure it works correctly."""
    
    if test_cases is None:
        # Define test cases based on patterns observed in the data
        test_cases = [
            # Cases likely to be PRESENT (based on training data patterns)
            {'age': 71, 'number': 3, 'start': 5, 'expected': 'present', 'reason': 'Young age, low start vertebra'},
            {'age': 158, 'number': 3, 'start': 14, 'expected': 'present', 'reason': 'Medium age, high start vertebra'},
            {'age': 15, 'number': 7, 'start': 2, 'expected': 'present', 'reason': 'Very young, many vertebrae, low start'},
            
            # Cases likely to be ABSENT
            {'age': 70, 'number': 2, 'start': 10, 'expected': 'absent', 'reason': 'Medium age, few vertebrae, mid start'},
            {'age': 210, 'number': 3, 'start': 12, 'expected': 'absent', 'reason': 'Old age, few vertebrae'},
            {'age': 100, 'number': 3, 'start': 17, 'expected': 'absent', 'reason': 'Medium age, high start vertebra'},
        ]
    
    print("\n" + "="*80)
    print("MODEL PREDICTION TESTING")
    print("="*80)
    
    for i, case in enumerate(test_cases, 1):
        # Prepare input
        input_data = np.array([[case['age'], case['number'], case['start']]])
        
        # Make prediction
        prediction_class = model.predict(input_data)[0]
        prediction_proba = model.predict_proba(input_data)[0]
        
        # Convert to string
        prediction_str = "present" if prediction_class == 1 else "absent"
        confidence = max(prediction_proba)
        
        # Display results
        print(f"\nTest Case {i}:")
        print(f"  Input: Age={case['age']}, Number={case['number']}, Start={case['start']}")
        print(f"  Expected: {case['expected']} ({case['reason']})")
        print(f"  Predicted: {prediction_str} (class: {prediction_class})")
        print(f"  Confidence: {confidence:.4f}")
        print(f"  Probabilities: Absent={prediction_proba[0]:.4f}, Present={prediction_proba[1]:.4f}")
        print(f"  Status: {'✓ CORRECT' if prediction_str == case['expected'] else '✗ INCORRECT'}")

# Test the model
test_model_predictions(model)

## 8. Sample Predictions from Training Data

In [ ]:
# Test with some actual samples from the dataset
print("\n" + "="*80)
print("PREDICTIONS ON ACTUAL TRAINING DATA SAMPLES")
print("="*80)

# Get some samples from both classes
present_samples = df[df['Kyphosis'] == 'present'].head(3)
absent_samples = df[df['Kyphosis'] == 'absent'].head(3)

print("\nTesting on PRESENT cases from training data:")
for idx, row in present_samples.iterrows():
    input_data = np.array([[row['Age'], row['Number'], row['Start']]])
    prediction_class = model.predict(input_data)[0]
    prediction_proba = model.predict_proba(input_data)[0]
    prediction_str = "present" if prediction_class == 1 else "absent"
    
    print(f"  Sample {idx}: Age={row['Age']}, Number={row['Number']}, Start={row['Start']}")
    print(f"    True: present, Predicted: {prediction_str}, Confidence: {max(prediction_proba):.4f}")
    print(f"    {'✓ CORRECT' if prediction_str == 'present' else '✗ INCORRECT'}")

print("\nTesting on ABSENT cases from training data:")
for idx, row in absent_samples.iterrows():
    input_data = np.array([[row['Age'], row['Number'], row['Start']]])
    prediction_class = model.predict(input_data)[0]
    prediction_proba = model.predict_proba(input_data)[0]
    prediction_str = "present" if prediction_class == 1 else "absent"
    
    print(f"  Sample {idx}: Age={row['Age']}, Number={row['Number']}, Start={row['Start']}")
    print(f"    True: absent, Predicted: {prediction_str}, Confidence: {max(prediction_proba):.4f}")
    print(f"    {'✓ CORRECT' if prediction_str == 'absent' else '✗ INCORRECT'}")

## 9. Save the Trained Model

In [ ]:
def save_model(model, filename='model.pkl'):
    """Save the trained model to disk."""
    try:
        joblib.dump(model, filename)
        print(f"\n✓ Model saved successfully as '{filename}'")
        
        # Verify the saved model by loading it
        loaded_model = joblib.load(filename)
        
        # Test that the loaded model works
        test_input = np.array([[71, 3, 5]])
        test_prediction = loaded_model.predict(test_input)[0]
        test_proba = loaded_model.predict_proba(test_input)[0]
        
        print(f"✓ Model verification successful")
        print(f"  Test prediction: {test_prediction} ({'present' if test_prediction == 1 else 'absent'})")
        print(f"  Test probabilities: {test_proba}")
        
    except Exception as e:
        print(f"✗ Error saving model: {e}")
        raise

# Save the model
save_model(model)

## 10. Summary and Conclusions

In [ ]:
print("\n" + "="*70)
print("TRAINING SUMMARY AND CONCLUSIONS")
print("="*70)

print(f"\n📊 Dataset Information:")
print(f"  • Total samples: {len(df)}")
print(f"  • Features: {list(X.columns)}")
print(f"  • Target classes: present (1), absent (0)")
print(f"  • Class distribution: {df['Kyphosis'].value_counts().to_dict()}")

print(f"\n🤖 Model Configuration:")
print(f"  • Algorithm: Decision Tree Classifier")
print(f"  • Max depth: 10")
print(f"  • Min samples split: 5")
print(f"  • Min samples leaf: 2")
print(f"  • Class weight: balanced (to handle imbalanced data)")

print(f"\n📈 Performance Metrics:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"  • Accuracy: {accuracy:.4f}")
print(f"  • Precision: {precision:.4f}")
print(f"  • Recall: {recall:.4f}")
print(f"  • F1-Score: {f1:.4f}")

print(f"\n🎯 Key Insights:")
print(f"  • Most important feature: {feature_importance.iloc[0]['feature']} ({feature_importance.iloc[0]['importance']:.4f})")
print(f"  • Model can predict both 'present' and 'absent' cases")
print(f"  • Class balancing improves minority class (present) prediction")

print(f"\n✅ Model Status: READY FOR DEPLOYMENT")
print(f"  • Model saved as 'model.pkl'")
print(f"  • Compatible with FastAPI backend")
print(f"  • Prediction format: 'present' (1) or 'absent' (0)")

print(f"\n🚀 Next Steps:")
print(f"  1. Start FastAPI server: python -m uvicorn main:app --host 127.0.0.1 --port 8000")
print(f"  2. Test predictions via API: POST to /predict endpoint")
print(f"  3. Open frontend in browser to use the web interface")

print("\n" + "="*70)